# Reading Eyelink Data

Imagine a simple memory task: on each trial, after a brief fixation period, we present a string of letters (e.g., XFABWS) and ask participants to memorize them. Then, after a retention interval, we present a probe letter (e.g., A) and ask if this letter belongs to the presented letters.  

While simple, the task is representative of a typical task used in psychological experiments: the repeated presentation of trials, with each trial consisting of several successive components.

The trial flow looks like this:

```{image} assets/trial.png
:alt: asc_file
:width: 800px
:align: center
```

## Read Data

What does the eye-tracking data in this task look like? Here, we will work with eye movement data recorded by the popular [Eyelink](https://www.sr-research.com/) system. Eyelink saves data in its proprietary `.edf` format, but it can be easily converted to a text file in `.asc` format. **PupEyes reads the converted `.asc` files so make sure you [convert your files](https://www.sr-research.com/support/thread-7674.html) beforehand**. 


An `.asc` file can be opened with a text editor. Here is a snippet of an example file:

```{image} assets/asc.png
:alt: asc_file
:width: 600px
:align: center
```

Most rows consist of raw gaze samples recorded at a certain sampling rate (e.g., 1000 Hz). The four columns indicate the timestamp, `x` and `y` coordinates, and pupil size, respectively.

Mixed with those raw gaze samples is information about fixations, saccades, and blinks automatically detected during recording. In this snippet, `SFIX` and `EFIX` indicate the start and the end of a fixation, respectively. `SSACC` and `ESACC` indicate the start and the end of a saccade, respectively.

Rows with an `MSG` prefix are event markers that the researcher sent during the task using custom code. These markers are important because they link data to task events. In our task, we sent a marker at the start and the end of each component within a trial, like below:

```{list-table}
:header-rows: 1

* - Marker
  - Event
  - Block
  - Trial
* - start
  - fixation
  - A
  - 4
* - end
  - fixation
  - A
  - 4
* - start
  - stimulus
  - A
  - 4
* - end
  - stimulus
  - A
  - 4
* - start
  - retention
  - A
  - 4
* - end
  - retention
  - A
  - 4
* - start
  - probe
  - A
  - 4
* - end
  - probe
  - A
  - 4
* - start
  - feedback
  - A
  - 4
* - end
  - feedback
  - A
  - 4
```

Understanding your event markers is critical for correctly parsing the raw data. Specifically, PupEyes needs to understand two things about your event markers:

- **The format of your event markers**. For example, if your event markers are like "start retention A 4", then the event markers consist of four parts: marker (start/end), the specific event, block ID, and trial ID. The delimiter is a space.

- **The notations for trial boundary**. In our case, a trial always starts with "start fixation X X" and ends with "end feedback X X". So the boundary is "start fixation" and "end feedback".

In [1]:
# file name
path = 'data/sub001.asc'

# event marker format, specified as a dictionary {name: data type}   
msg_format = {'marker':str, 'event':str, 'block':str, 'trial':int} # e.g., start retention A 4
delimiter = ' ' # delimiter for messages

# start and stop notations for each trial
start_msg = 'start fixation'
stop_msg = 'end feedback'

# If you have any constant columns that you want to add
add_cols = {'subject':'sub001', 'condition': 'low'}

Once you have the required information, simply pass it to `EyelinkReader`:

In [2]:
import pupeyes as pe

raw = pe.EyelinkReader(path=path, 
                       start_msg=start_msg, 
                       stop_msg=stop_msg, 
                       msg_format=msg_format, 
                       delimiter=delimiter, 
                       add_cols=add_cols
                      ) 

Reading ASC file data/sub001.asc:   0%|          | 0/172262 [00:00<?, ?it/s]

```{tip}
- When building your task, consider where you need an event marker and how to format it so that the messages can be correctly parsed. 
- If your task consists of multiple events within each trial (like the one here), it may make sense to mark each event's start and end. Better safe than sorry!
- If you only want data for a part of the trial, simply change `start_msg` and/or `stop_msg`. 
```

## Get Gaze Samples

Once we create an `EyelinkReader` instance, getting raw gaze samples is easy:

In [3]:
samples = raw.get_samples()
samples.head(5) # showing the first 5 rows

,trialtime,trackertime,x,y,pp,msg,msgtime,marker,event,block,trial,subject,condition
0,0,3148741,856.9,424.4,5545,start fixation A 1,3148741,start,fixation,A,1,sub001,low
1,1,3148742,857.5,425.9,5546,start fixation A 1,3148741,start,fixation,A,1,sub001,low
2,2,3148743,858.0,427.2,5547,start fixation A 1,3148741,start,fixation,A,1,sub001,low
3,3,3148744,858.2,427.2,5547,start fixation A 1,3148741,start,fixation,A,1,sub001,low
4,4,3148745,857.9,427.1,5545,start fixation A 1,3148741,start,fixation,A,1,sub001,low


`.get_samples()` returns a `pandas` dataframe with each row indicating one gaze sample.

1. `trialtime`: Timestamps during a trial. Reset to 0 for every new trial (i.e., what the user provides as the start message).
2. `trackertime`: Timestamps as recorded by Eyelink. Does not reset.
3. `x` and `y`: gaze position in Eyelink coordinates (pixels with (0,0) being the top-left corner).
4. `pp`: pupil size in arbitrary unit (either AREA of DIAMETER, depending on Eyelink setting).
5. `msg`: event marker message.
6. `msgtime`: the timestamp of the event marker message.
7. `marker`, `event`, `block`, and `trial`: parsed event markers according to user specification.
8. `subject` and `condition`: constant columns added by user.

**Now you have the data in an analysis-friendly format, you are set to start whatever analysis you want. Check out the [**vast data processing methods**](https://pandas.pydata.org/docs/index.html) `pandas` offers.**

Note that PupEyes comes with comprehensive functionalities for pupil size preprocessing, which will be introduced in [Pupil Preprocessing](pupil_preproc.ipynb).

## Get Fixation Data

Just like how we got the gaze samples:

In [4]:
fixations = raw.get_fixations()
fixations.head(5) # showing the first 5 rows

,eye,starttime,endtime,duration,endx,endy,msg,msgtime,marker,event,block,trial,subject,condition
0,R,3148925,3149062,138,635.4,463.2,start fixation A 1,3148741,start,fixation,A,1,sub001,low
1,R,3149898,3149941,44,655.5,1233.8,start stimulus A 1,3149725,start,stimulus,A,1,sub001,low
2,R,3149984,3150025,42,733.8,691.1,start stimulus A 1,3149725,start,stimulus,A,1,sub001,low
3,R,3150067,3150187,121,770.1,438.0,start stimulus A 1,3149725,start,stimulus,A,1,sub001,low
4,R,3150205,3150355,151,717.9,486.4,start stimulus A 1,3149725,start,stimulus,A,1,sub001,low


`.get_fixations()` returns a `pandas` dataframe with each row showing one fixation, as detected by Eyelink.
1. `eye`: which eye is recorded for this fixation.
2. `starttime` and `endtime`: Start and end times for this fixation.
3. `duration`: Fixation duration in milliseconds.
4. `endx` and `endy`: Fixation position in Eyelink coordinates (pixels with (0,0) being the top-left corner).

The rest are the same as the pupil size data.

## Get Saccade Data

In [5]:
saccades = raw.get_saccades()
saccades.head(5) # showing the first 5 rows

,eye,starttime,endtime,duration,startx,starty,endx,endy,ampl,pv,msg,msgtime,srt,marker,event,block,trial,subject,condition
0,R,3148894,3148924,31,861.1,432.3,632.5,466.4,3.61,223,start fixation A 1,3148741,153,start,fixation,A,1,sub001,low
1,R,3149063,3149095,33,632.9,465.2,965.4,473.1,5.20,299,start fixation A 1,3148741,322,start,fixation,A,1,sub001,low
2,R,3149846,3149897,52,958.4,495.0,667.1,1211.3,11.94,460,start stimulus A 1,3149725,121,start,stimulus,A,1,sub001,low
3,R,3149942,3149983,42,649.1,1231.9,741.0,690.7,8.44,528,start stimulus A 1,3149725,217,start,stimulus,A,1,sub001,low
4,R,3150026,3150066,41,727.7,688.8,772.0,461.0,3.63,398,start stimulus A 1,3149725,301,start,stimulus,A,1,sub001,low


`.get_saccades()` returns a `pandas` dataframe with each row showing one saccade, as detected by Eyelink.
1. `eye`: which eye is recorded for this saccade.
2. `starttime` and `endtime`: Start and end times for this saccade.
3. `duration`: Saccade duration in milliseconds.
4. `startx`, `starty`, `endx` and `endy`: Saccade start and end positions in Eyelink coordinates (pixels with (0,0) being the top-left corner).
5. `ampl`: Saccade amplitude (i.e., total visual angle covered in the saccade).
6. `pv`: Saccade peak velocity.
7. `srt`: Saccade latency, calculated as the difference between saccade start time `starttime` and message timestamp `msgtime`.

The rest are the same as the pupil size data.

## Get Blinks

In [6]:
blinks = raw.get_blinks()
blinks.head(5) # showing the first 5 rows

,eye,starttime,endtime,duration,msg,msgtime,marker,event,block,trial
0,R,3153626,3153732,107,start retention A 1,3152725,start,retention,A,1
1,R,3155486,3155664,179,start retention A 1,3152725,start,retention,A,1
2,R,3156508,3156675,168,start retention A 1,3152725,start,retention,A,1
3,R,3158260,3158409,150,start retention A 1,3152725,start,retention,A,1
4,R,3160026,3160238,213,start retention A 1,3152725,start,retention,A,1


`.get_blinks()` returns a `pandas` dataframe with each row showing one blink, as detected by Eyelink.
1. `eye`: which eye is recorded for this blink.
2. `starttime` and `endtime`: Start and end times for this blink.
3. `duration`: Saccade duration in milliseconds.

The rest are the same as the pupil size data.

## Metadata

PupEyes also stores the metadata, which can be useful for checking calibration quality, sampling rate, pupil size unit, etc. 

For example, we can see that two 5-point calibrations were performed. The first resulted in an aborted validation, and the second resulted in a successful validation.

In [7]:
raw.metadata

{'CALIBRATION_TYPE': ['HV5', 'HV5'],
 'CALIBRATION_EYE': ['R', 'R'],
 'CALIBRATION_RESULT': ['GOOD', 'GOOD'],
 'VALIDATION_TYPE': ['ABORTED', 'HV5'],
 'VALIDATION_EYE': ['R', 'R'],
 'VALIDATION_RESULT': ['ABORTED', 'GOOD'],
 'TRACKING_MODE': ['CR'],
 'SAMPLING_RATE': ['1000'],
 'FILE_SAMPLE_FILTER': ['2'],
 'LINK_SAMPLE_FILTER': ['0'],
 'EYE_RECORDED': ['R'],
 'MOUNT_CONFIG': ['MTABLER'],
 'GAZE_COORDS': [['0.00', '0.00', '1920.00', '1080.00']],
 'PUPIL': ['DIAMETER'],
 'PUPIL_TRACKING_ALGORITHM': ['CENTROID']}

## Q & A

**What If I have Multiple Participants?**

No problem. You can do it with an old-fashioned `for` loop.

Suppose we have files named `sub001.asc`, `sub002.asc`...etc. We can use a wildcard to loop through all files ending with `.asc`, extract data for each, and then combine them into a single dataframe.

In [8]:
from glob import glob
from os.path import basename

# empty list to store individual subject's data
list_of_data = [] 

# loop through files based on specified file name pattern
for path in glob('data/*.asc'):
    
    # extract subject id from filename
    participant = basename(path)[:-4]
    
    # load Eyelink asc file
    raw_subject = pe.EyelinkReader(path=path, start_msg=start_msg, stop_msg=stop_msg, msg_format=msg_format, delimiter=delimiter, add_cols={'participant':participant})

    # get fixations for this subject
    fixations_subject = raw_subject.get_fixations()
    
    # append data to the list
    list_of_data.append(fixations_subject)

Reading ASC file data\sub001.asc:   0%|          | 0/172262 [00:00<?, ?it/s]

Reading ASC file data\sub002.asc:   0%|          | 0/177914 [00:00<?, ?it/s]

Reading ASC file data\sub003.asc:   0%|          | 0/164484 [00:00<?, ?it/s]

Reading ASC file data\sub004.asc:   0%|          | 0/172636 [00:00<?, ?it/s]

Then, you just need to concatenate the list of data into a single `pandas` dataframe:

In [9]:
import pandas as pd

# concatenate
fixations_all = pd.concat(list_of_data, ignore_index=True) # ignore_index = True so that index match the number of rows!
fixations_all

,eye,starttime,endtime,duration,endx,endy,msg,msgtime,marker,event,block,trial,participant
0,R,3148925,3149062,138,635.4,463.2,start fixation A 1,3148741,start,fixation,A,1,sub001
1,R,3149898,3149941,44,655.5,1233.8,start stimulus A 1,3149725,start,stimulus,A,1,sub001
2,R,3149984,3150025,42,733.8,691.1,start stimulus A 1,3149725,start,stimulus,A,1,sub001
3,R,3150067,3150187,121,770.1,438.0,start stimulus A 1,3149725,start,stimulus,A,1,sub001
4,R,3150205,3150355,151,717.9,486.4,start stimulus A 1,3149725,start,stimulus,A,1,sub001
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1310,R,3227521,3227696,176,881.1,589.9,start probe A 10,3226693,start,probe,A,10,sub004
1311,R,3227711,3228124,414,928.1,584.4,start probe A 10,3226693,start,probe,A,10,sub004
1312,R,3228706,3229031,326,879.0,574.1,start feedback A 10,3228381,start,feedback,A,10,sub004
1313,R,3229040,3229076,37,853.2,575.2,start feedback A 10,3228381,start,feedback,A,10,sub004


**What About Other Eye-trackers?**

Our lab exclusively uses Eyelink, which is why I have prioritized it for now. I'm happy to look into it if there is a need for similar functionalities for another eye-tracker.

Even if you are using another eye-tracker, PupEyes may still be useful to you for pupil preprocessing, data visualization, AOI drawing, and other purposes. Check out tutorials on those topics!

```{seealso}
A comprehensive introduction to Eyelink data: <https://www.sr-research.com/support/thread-7675.html>

That whole support forum is a trove of treasures, and I recommend it to anyone who wishes to do eye-tracking with Eyelink.
```